In [1]:
import time
import sys
import torch
import theseus as th
import open3d as o3d

sys.path.append("../")
import cv2 as cv
import numpy as np
from pathlib import Path

from scipy.spatial.transform import Rotation

from dataclasses import dataclass

import matplotlib.pyplot as plt
import plotly.graph_objects as go

from sulllam.utils.ros import ROSPublisherWrapper
from sulllam.localization.extraction.orb import ORBFeatureExtractor, ORBConfigs
from sulllam.localization.matching.bf import BFMatcherConfig, BFFeatureMatcher
from sulllam.localization.pose_estimation.eight_point_estimator import EightPointEstimatorConfig, EightPointPoseEstimator

/home/zhukowych/Projects/ucu/MMML/SULLLAM/submodules/theseus/theseus/optimizer/autograd/cholmod_sparse_autograd.py:14: UserWarning: Couldn't import skparse.cholmod. Cholmod solver won't work.
  warnings.warn("Couldn't import skparse.cholmod. Cholmod solver won't work.")
/home/zhukowych/Projects/ucu/MMML/SULLLAM/submodules/theseus/theseus/optimizer/linear/cholmod_sparse_solver.py:19: UserWarning: Couldn't import skparse.cholmod. Cholmod solver won't work.
  warnings.warn("Couldn't import skparse.cholmod. Cholmod solver won't work.")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/mnt/disk2/.virtualenvs/.sullam/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Images:

In [ ]:
image_dir = Path("/home/zhukowych/Projects/ucu/MMML/SULLLAM/data/itspace")
images = [cv.imread(image_path) for image_path in sorted(image_dir.glob("*.png"))[::10]]
len(images)

144

: 

: 

: 

Camera calibration:

In [ ]:
K = np.array([[533.340727445877, 0.0, 254.64689387916482],
              [0.0, 533.2556495307942, 256.4835490935692],
              [0.0, 0.0, 1.0]])

xi = np.array([[1.73241756065]])

D = np.array([-0.05972430882700243, 0.17468739202093328, 0.000737218969875311, 0.000574074894976456])

fx, fy = 2740.0, 2740.0
cx, cy = 2016.0, 1512.0

# Define the K matrix
K = np.array([
    [1200.0,    0.0,  960.0],
    [   0.0, 1200.0,  540.0],
    [   0.0,    0.0,    1.0]
])

: 

: 

: 

Initialize all components

In [ ]:
def Rt_to_T(R, t):
    """Converts a 3x3 rotation matrix and 3x1 translation vector into a 4x4 transform."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    return T

: 

: 

: 

In [ ]:
class PointMap:
    def __init__(self, max_points: int = 1_000_000, max_observations: int = 5_000_000):
        self.points_3d      = np.zeros((max_points, 3),      dtype=np.float64)
        self.point_colors   = np.zeros((max_points, 3),      dtype=np.uint8)
        self.num_points     = 0
 
        self.observations   = np.zeros((max_observations, 4), dtype=np.float64)
        self.num_observations = 0
 
        # Inverted index: kf_idx  -> list[observation row index]
        # Allows O(window_size × pts_per_frame) lookup instead of O(all_obs).
        self._kf_to_obs: dict[int, list[int]] = {}
 
    # ------------------------------------------------------------------
    def add_point(self, xyz, color=None) -> int:
        pt_id = self.num_points
        self.points_3d[pt_id] = xyz
        if color is not None:
            self.point_colors[pt_id] = np.clip(color, 0, 255).astype(np.uint8)
        self.num_points += 1
        return pt_id
 
    def add_observation(self, point_id: int, keyframe_id: int, uv_coords) -> int:
        obs_id = self.num_observations
        self.observations[obs_id, 0] = point_id
        self.observations[obs_id, 1] = keyframe_id
        self.observations[obs_id, 2:] = uv_coords          # stored as [u, v] = [col, row]
        self._kf_to_obs.setdefault(keyframe_id, []).append(obs_id)
        self.num_observations += 1
        return obs_id
 
    # ------------------------------------------------------------------
    def observations_for_keyframes(self, kf_ids: set[int]) -> list[int]:
        """Return observation row indices that belong to any of the given keyframe ids."""
        obs_indices = []
        for kf_id in kf_ids:
            obs_indices.extend(self._kf_to_obs.get(kf_id, []))
        return obs_indices
 
    # ------------------------------------------------------------------
    def to_open3d(self):
        import open3d as o3d
        pcd = o3d.geometry.PointCloud()
        points = self.points_3d[: self.num_points]
        colors = self.point_colors[: self.num_points].astype(np.float64) / 255.0
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        return pcd
 
    def save_pointcloud(self, filename: str):
        import open3d as o3d
        pcd = self.to_open3d()
        o3d.io.write_point_cloud(str(filename), pcd)

class Keyframe:
    def __init__(self, idx, keypopints, descriptors, pose):
        self.idx = idx
        self.keypoints = keypopints 
        self.descriptors = descriptors
        self.pose = pose
    
    @property
    def R(self):
        return self.pose[:3, :3]
    
    @property
    def t(self):
        return self.pose[:3, 3]


class Mapper:
    def __init__(self):
        self.pointmap = PointMap()
        self.keyframes = []
        
    def add_keyframe(self, keyframe):
        self.keyframes.append(keyframe)
        
    @property
    def current_keyframe(self):
        return self.keyframes[-1] if self.keyframes else None
        
    @property
    def previous_keyframe(self):
        return self.keyframes[-2] if len(self.keyframes) > 1 else None

: 

: 

: 

In [ ]:

def reproj_error_opt_cam(optim_vars, aux_vars):
    """Cost function when both Camera Pose and 3D Point are being optimized."""
    pose, point = optim_vars
    K_vec, uv = aux_vars
    
    # transform_to natively handles Theseus variables and returns a PyTorch tensor
    pt_cam = pose.transform_to(point)
    
    # Extract the underlying PyTorch tensors from Theseus Auxiliary Variables
    K_tensor = K_vec.tensor
    uv_tensor = uv.tensor
    
    fx, fy, cx, cy = K_tensor[:, 0], K_tensor[:, 1], K_tensor[:, 2], K_tensor[:, 3]
    
    # Perspective division
    z = pt_cam[:, 2] + 1e-8
    x = pt_cam[:, 0] / z
    y = pt_cam[:, 1] / z
    
    # Project to pixel coordinates
    u = fx * x + cx
    v = fy * y + cy
    pred = torch.stack([u, v], dim=1)
    
    # Now both are pure PyTorch Tensors
    return pred - uv_tensor


def reproj_error_fixed_cam(optim_vars, aux_vars):
    """Cost function when the Camera Pose is fixed (gauge anchoring)."""
    point, = optim_vars
    pose, K_vec, uv = aux_vars
    
    pt_cam = pose.transform_to(point)
    
    # Extract the underlying PyTorch tensors
    K_tensor = K_vec.tensor
    uv_tensor = uv.tensor
    
    fx, fy, cx, cy = K_tensor[:, 0], K_tensor[:, 1], K_tensor[:, 2], K_tensor[:, 3]
    
    z = pt_cam[:, 2] + 1e-8
    x = pt_cam[:, 0] / z
    y = pt_cam[:, 1] / z
    
    u = fx * x + cx
    v = fy * y + cy
    pred = torch.stack([u, v], dim=1)
    
    return pred - uv_tensor


def local_bundle_adjustment(mapper, K, window_size=5, huber_radius=2.0):
    """
    Performs Local BA on the last N keyframes and their observed points.
    """
    print(f"\n[BA DEBUG] --- Starting Local Bundle Adjustment ---")
    print(f"[BA DEBUG] Total keyframes in mapper: {len(mapper.keyframes)}")
    
    if len(mapper.keyframes) < 2:
        print("[BA DEBUG] Not enough frames to optimize. Exiting BA.")
        return  # Not enough frames to optimize

    # 1. Identify local keyframes
    local_kfs = mapper.keyframes[-window_size:]
    local_kf_ids = [kf.idx for kf in local_kfs]
    print(f"[BA DEBUG] Local window size: {window_size}")
    print(f"[BA DEBUG] Local keyframe IDs: {local_kf_ids}")
    
    # 2. Extract observations related to the local window
    all_obs = mapper.pointmap.observations[:mapper.pointmap.num_observations]
    print(f"[BA DEBUG] Total observations in pointmap: {len(all_obs)}")
    
    # Find all points observed by our local keyframes
    mask = np.isin(all_obs[:, 1], local_kf_ids)
    local_point_ids = np.unique(all_obs[mask, 0]).astype(int)
    
    if len(local_point_ids) == 0:
        print("[BA DEBUG] No local points found for the current window. Exiting BA.")
        return
        
    print(f"[BA DEBUG] Found {len(local_point_ids)} unique 3D points observed in the local window.")
    
    # Get ALL observations of these local points (even from older, fixed keyframes)
    point_mask = np.isin(all_obs[:, 0], local_point_ids)
    ba_obs = all_obs[point_mask]
    ba_kf_ids = np.unique(ba_obs[:, 1]).astype(int)
    
    print(f"[BA DEBUG] Gathered {len(ba_obs)} total observations for these points.")
    print(f"[BA DEBUG] These observations span {len(ba_kf_ids)} unique keyframes: {ba_kf_ids.tolist()}")

    # 3. Setup Theseus Objective
    print("[BA DEBUG] Setting up Theseus Objective...")
    objective = th.Objective(dtype=torch.float64)
    weight = th.ScaleCostWeight(torch.tensor(1.0, dtype=torch.float64))
    
    # Log radius for Huber loss to handle outlier matches
    log_loss_radius = th.Vector(
        tensor=torch.tensor([[np.log(huber_radius)]], dtype=torch.float64), 
        name="log_loss_radius"
    )

    # 4. Create Variables
    se3_vars = {}
    pt_vars = {}

    # Initialize camera pose variables (World to Camera)
    for kf_id in ba_kf_ids:
        kf = next(k for k in mapper.keyframes if k.idx == kf_id)
        pose_tensor = torch.from_numpy(kf.pose[:3, :]).unsqueeze(0).double()
        se3_vars[kf_id] = th.SE3(tensor=pose_tensor, name=f"cam_{kf_id}")

    # Initialize 3D point variables
    for pt_id in local_point_ids:
        pt3d = mapper.pointmap.points_3d[pt_id]
        pt_tensor = torch.from_numpy(pt3d).unsqueeze(0).double()
        pt_vars[pt_id] = th.Point3(tensor=pt_tensor, name=f"pt_{pt_id}")

    print(f"[BA DEBUG] Created {len(se3_vars)} SE3 variables and {len(pt_vars)} Point3 variables.")

    # Intrinsic parameters mapped to a tensor: [fx, fy, cx, cy]
    K_tensor = torch.tensor([[K[0,0], K[1,1], K[0,2], K[1,2]]], dtype=torch.float64)

    # 5. Fix gauge freedom (anchor the oldest camera in the optimization)
    fixed_kf_ids = set(ba_kf_ids) - set(local_kf_ids)
    if len(fixed_kf_ids) == 0:
        fixed_kf_ids.add(local_kf_ids[0])
        print(f"[BA DEBUG] No older cameras found. Anchoring the first local frame: {local_kf_ids[0]}")
    
    print(f"[BA DEBUG] Fixed (Anchored) Keyframes: {list(fixed_kf_ids)}")
    print(f"[BA DEBUG] Optimized Keyframes: {list(set(ba_kf_ids) - fixed_kf_ids)}")

    # 6. Build the Optimization Graph
    fixed_cost_count = 0
    opt_cost_count = 0
    
    for o in ba_obs:
        pt_id, kf_id = int(o[0]), int(o[1])
        uv_tensor = torch.tensor([[o[2], o[3]]], dtype=torch.float64)
        
        uv_var = th.Vector(tensor=uv_tensor, name=f"uv_{pt_id}_{kf_id}")
        K_var = th.Vector(tensor=K_tensor, name=f"K_{pt_id}_{kf_id}")
        
        cam_var = se3_vars[kf_id]
        pt_var = pt_vars[pt_id]
        
        if kf_id in fixed_kf_ids:
            # Camera is fixed, only optimize the point
            cost_fn = th.AutoDiffCostFunction(
                optim_vars=[pt_var],
                err_fn=reproj_error_fixed_cam,
                dim=2,
                aux_vars=[cam_var, K_var, uv_var],
                cost_weight=weight,
                name=f"cost_{pt_id}_{kf_id}"
            )
            fixed_cost_count += 1
        else:
            # Optimize both Camera and Point
            cost_fn = th.AutoDiffCostFunction(
                optim_vars=[cam_var, pt_var],
                err_fn=reproj_error_opt_cam,
                dim=2,
                aux_vars=[K_var, uv_var],
                cost_weight=weight,
                name=f"cost_{pt_id}_{kf_id}"
            )
            opt_cost_count += 1
            
        robust_cost = th.RobustCostFunction(
            cost_function=cost_fn,
            loss_cls=th.HuberLoss,
            log_loss_radius=log_loss_radius,
            name=f"robust_cost_{pt_id}_{kf_id}"
        )
        objective.add(robust_cost)

    print(f"[BA DEBUG] Graph built. Added {fixed_cost_count} fixed-cam edges and {opt_cost_count} opt-cam edges.")

    # 7. Run Optimizer
    
    print("[BA DEBUG] Updating objective...")
    objective.update()

    # --- FIX 1: Use error_metric().sum() instead of error_squared_norm() ---
    initial_error = objective.error_metric().sum().item()
    print(f"[BA DEBUG] Initial Squared Error: {initial_error:.4f}")

    optimizer = th.LevenbergMarquardt(
        objective,
        max_iterations=5,  # Keep iterations low for real-time performance
        step_size=0.5
    )
    
    print("[BA DEBUG] Starting Levenberg-Marquardt optimization loop...")
    info = optimizer.optimize()
    
    # --- FIX 2: Apply the exact same change here ---
    final_error = objective.error_metric().sum().item()
    
    print(f"[BA DEBUG] Optimization Finished!")
    print(f"  -> Status: {info.status[0]}")
    print(f"  -> Iterations executed: {info.converged_iter[0].item() + 1}")
    print(f"  -> Final Squared Error: {final_error:.4f} (Delta: {initial_error - final_error:.4f})")

: 

: 

: 

In [ ]:
ros_publisher = ROSPublisherWrapper()

: 

: 

: 

In [ ]:
orb_config = ORBConfigs()
orb_extrator = ORBFeatureExtractor()

bf_config = BFMatcherConfig()
bf_matcher = BFFeatureMatcher(config=bf_config)

eight_point_config = EightPointEstimatorConfig(K=K)
eight_point_pose_estimator = EightPointPoseEstimator(config=eight_point_config)


mapper = Mapper()

: 

: 

: 

In [ ]:

CLOUDS_DIR = Path("clouds")
MAX_REPROJ_ERROR = 2.0 
MAX_DEPTH = 50

for item in CLOUDS_DIR.iterdir():
    if item.is_file():
        item.unlink()


trajectory = [
    np.array([0, 0, 0]).T
]

R_global = np.array([
    [0, 0, 1],
    [1, 0, 0],
    [0, 1, 0]
])
t_global = np.zeros(3)

previous_keypoints, previous_descriptors = orb_extrator.extract(images[0])

initial_keyframe = Keyframe(
    idx=0,
    keypopints=previous_keypoints,
    descriptors=previous_descriptors,
    pose=Rt_to_T(R_global, t_global)
)

mapper.add_keyframe(initial_keyframe)

# --- Main Processing Loop ---

for i in range(1, len(images)):
    time.sleep(1)

    current_keypoints, current_descriptors = orb_extrator.extract(images[i])
    matches = bf_matcher.match(previous_descriptors, current_descriptors)

    previous_keypoints_sorted = np.array([previous_keypoints[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    current_keypoints_sorted = np.array([current_keypoints[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    undistorted_points1 = previous_keypoints_sorted
    undistorted_points2 = current_keypoints_sorted

    estimate = eight_point_pose_estimator.estimate(previous_keypoints_sorted, current_keypoints_sorted)

    R = estimate['R']
    t = estimate['t'].reshape(-1)

    inliers_mask = estimate['inliers_mask']

    # Update Global Pose
    t_global = R @ t_global + t
    R_global = R @ R_global

    current_keyframe = Keyframe(
        idx=i,
        keypopints=current_keypoints,
        descriptors=current_descriptors,
        pose=Rt_to_T(R_global, t_global)
    )

    mapper.add_keyframe(current_keyframe)

    # --- Point Triangulation ---

    R_prev = mapper.previous_keyframe.R
    t_prev = mapper.previous_keyframe.t

    R_curr = mapper.current_keyframe.R
    t_curr = mapper.current_keyframe.t

    P1 = K @ np.hstack((R_prev, t_prev.reshape(3, 1)))
    P2 = K @ np.hstack((R_curr, t_curr.reshape(3, 1)))

    pts1_inliers = previous_keypoints_sorted[inliers_mask.ravel() == 1].reshape(-1, 2).T
    pts2_inliers = current_keypoints_sorted[inliers_mask.ravel() == 1].reshape(-1, 2).T

    points_4d = cv.triangulatePoints(P1, P2, pts1_inliers, pts2_inliers)

    points_3d = points_4d[:3, :] / points_4d[3, :]
    points_3d = points_3d.T

    rvec_prev, _ = cv.Rodrigues(R_prev)
    rvec_curr, _ = cv.Rodrigues(R_curr)

    proj_cam1, _ = cv.projectPoints(points_3d, rvec_prev, t_prev, K, None)
    proj_cam2, _ = cv.projectPoints(points_3d, rvec_curr, t_curr, K, None)

    err1 = np.linalg.norm(proj_cam1.reshape(-1, 2) - pts1_inliers.T, axis=1)
    err2 = np.linalg.norm(proj_cam2.reshape(-1, 2) - pts2_inliers.T, axis=1)

    pts3d_cam1 = (R_prev @ points_3d.T + t_prev.reshape(3, 1)).T
    pts3d_cam2 = (R_curr @ points_3d.T + t_curr.reshape(3, 1)).T

    current_image_rgb = cv.cvtColor(images[i], cv.COLOR_BGR2RGB)

    for idx, pt3d in enumerate(points_3d):
        if pts3d_cam1[idx, 2] <= 0.0 or pts3d_cam2[idx, 2] <= 0.0:
            continue

        if pts3d_cam1[idx, 2] >= MAX_DEPTH or pts3d_cam2[idx, 2] >= MAX_DEPTH:
            continue
            
        if err1[idx] > MAX_REPROJ_ERROR or err2[idx] > MAX_REPROJ_ERROR:
            continue

        u, v = pts2_inliers[0, idx].astype(int), pts2_inliers[1, idx].astype(int)
        
        if 0 <= v < current_image_rgb.shape[0] and 0 <= u < current_image_rgb.shape[1]:
            point_color = current_image_rgb[v, u]
            point_id = mapper.pointmap.add_point(pt3d, color=point_color)

            uv_prev = pts1_inliers[:, idx]
            uv_curr = pts2_inliers[:, idx]
            
            mapper.pointmap.add_observation(point_id, mapper.previous_keyframe.idx, uv_prev)
            mapper.pointmap.add_observation(point_id, mapper.current_keyframe.idx, uv_curr)

    # --- Bundle Adjustment ---

    if i % 10 == 0:

        local_bundle_adjustment(mapper, K, window_size=5)

        # ── Fix: propagate LBA-corrected pose back to the running accumulators
        # so the next relative-pose composition starts from the right place.
        R_global = mapper.current_keyframe.R.copy()
        t_global = mapper.current_keyframe.t.copy()


    R_global = mapper.current_keyframe.R
    t_global = mapper.current_keyframe.t

    # --- Logging / Visualizations ---

    camera_position_world = -R_global.T @ t_global
    trajectory.append(camera_position_world)

    current_orientation = Rotation.from_matrix(R_global).as_quat()
    current_image_pair = cv.hconcat([images[i-1], images[i]])
    images_with_matches = cv.drawMatches(images[i-1], previous_keypoints, images[i], current_keypoints, matches, None)

    # ros_publisher.publish_pose(trajectory[-1], current_orientation)

    optimized_translations = []
    optimized_orientations = []

    for kf in mapper.keyframes:
        cam_pos_world = -kf.R.T @ kf.t
        optimized_translations.append(cam_pos_world)
        
        cam_quat_world = Rotation.from_matrix(kf.R.T).as_quat()
        optimized_orientations.append(cam_quat_world)

    ros_publisher.publish_trajectory(optimized_translations, optimized_orientations)

    ros_publisher.publish_current_pair(current_image_pair)
    ros_publisher.publish_current_matches(images_with_matches)
    ros_publisher.publish_pointcloud(
        mapper.pointmap.points_3d[:mapper.pointmap.num_points],
        mapper.pointmap.point_colors[:mapper.pointmap.num_points],
    )

    mapper.pointmap.save_pointcloud(CLOUDS_DIR / f"{i}.ply")

    # Advance descriptors for next iteration
    previous_keypoints, previous_descriptors = current_keypoints, current_descriptors

trajectory = np.array(trajectory)

[0. 0. 0.]
[0.04579237 0.00908366 0.99890968]
[-0.36010526 -0.02183969  1.91230492]
[-0.18038001 -0.44120812  2.80215263]
[-0.42133502 -0.49827206  3.77100989]
[-0.56430408 -0.57298897  4.75791274]
[-1.2524827  -0.56974953  5.48344682]
[-1.45205301 -0.6440786   6.46050714]
[-1.83892091 -0.71468203  7.37993543]
[-1.70111627 -0.82887379  8.36379016]
[-2.01178871 -0.9098327   9.31085314]

[BA DEBUG] --- Starting Local Bundle Adjustment ---
[BA DEBUG] Total keyframes in mapper: 11
[BA DEBUG] Local window size: 5
[BA DEBUG] Local keyframe IDs: [6, 7, 8, 9, 10]
[BA DEBUG] Total observations in pointmap: 2762
[BA DEBUG] Found 833 unique 3D points observed in the local window.
[BA DEBUG] Gathered 1666 total observations for these points.
[BA DEBUG] These observations span 6 unique keyframes: [5, 6, 7, 8, 9, 10]
[BA DEBUG] Setting up Theseus Objective...
[BA DEBUG] Created 6 SE3 variables and 833 Point3 variables.
[BA DEBUG] Fixed (Anchored) Keyframes: [np.int64(5)]
[BA DEBUG] Optimized Keyfram

/home/zhukowych/Projects/ucu/MMML/SULLLAM/submodules/theseus/theseus/optimizer/optimizer.py:43: UserWarning: Vectorization is off by default when not running from TheseusLayer. Using TheseusLayer is the recommended way to run our optimizers.
  warnings.warn(


[BA DEBUG] Optimization Finished!
  -> Status: NonlinearOptimizerStatus.MAX_ITERATIONS
  -> Iterations executed: 0
  -> Final Squared Error: 8891.4655 (Delta: 389341.3246)
[0. 0. 0.]
[0.04579237 0.00908366 0.99890968]
[-0.36010526 -0.02183969  1.91230492]
[-0.18038001 -0.44120812  2.80215263]
[-0.42133502 -0.49827206  3.77100989]
[-0.56430408 -0.57298897  4.75791274]
[-1.2524827  -0.56974953  5.48344682]
[-1.45205301 -0.6440786   6.46050714]
[-1.83892091 -0.71468203  7.37993543]
[-1.70111627 -0.82887379  8.36379016]
[-2.01178871 -0.9098327   9.31085314]
[0. 0. 0.]
[0.04579237 0.00908366 0.99890968]
[-0.36010526 -0.02183969  1.91230492]
[-0.18038001 -0.44120812  2.80215263]
[-0.42133502 -0.49827206  3.77100989]
[-0.56430408 -0.57298897  4.75791274]
[-1.2524827  -0.56974953  5.48344682]
[-1.45205301 -0.6440786   6.46050714]
[-1.83892091 -0.71468203  7.37993543]
[-1.70111627 -0.82887379  8.36379016]
[-2.01178871 -0.9098327   9.31085314]
[-2.07937422 -1.00516672 10.30400148]
[-1.83802581 -

/home/zhukowych/Projects/ucu/MMML/SULLLAM/submodules/theseus/theseus/optimizer/optimizer.py:43: UserWarning: Vectorization is off by default when not running from TheseusLayer. Using TheseusLayer is the recommended way to run our optimizers.
  warnings.warn(


[BA DEBUG] Optimization Finished!
  -> Status: NonlinearOptimizerStatus.MAX_ITERATIONS
  -> Iterations executed: 0
  -> Final Squared Error: 19277.6217 (Delta: 882994.2259)
[0. 0. 0.]
[0.04579237 0.00908366 0.99890968]
[-0.36010526 -0.02183969  1.91230492]
[-0.18038001 -0.44120812  2.80215263]
[-0.42133502 -0.49827206  3.77100989]
[-0.56430408 -0.57298897  4.75791274]
[-1.2524827  -0.56974953  5.48344682]
[-1.45205301 -0.6440786   6.46050714]
[-1.83892091 -0.71468203  7.37993543]
[-1.70111627 -0.82887379  8.36379016]
[-2.01178871 -0.9098327   9.31085314]
[-2.07937422 -1.00516672 10.30400148]
[-1.83802581 -1.4158783  11.18324374]
[-1.06882669 -1.62771585 11.78611815]
[-1.54746574 -1.68981387 12.66193119]
[-1.60577297 -1.76548701 13.65735765]
[-2.19524879 -1.82469247 14.46297107]
[-1.62573038 -1.99855164 15.26635253]
[-1.57422981 -2.46125215 16.15137   ]
[-1.62255176 -2.58375604 17.14266097]
[-1.42082603 -2.59235054 18.12206532]
[0. 0. 0.]
[0.04579237 0.00908366 0.99890968]
[-0.36010526 

: 

: 

: 

In [ ]:
ros_publisher.shutdown()

[ROS] Node shut down.


: 

: 

: 